# Test-set MAE histograms

This notebook reproduces the sample-wise mean absolute error (MAE) histograms for the $n=4,6,8$ PNN surrogate models. Each input file contains the complete two-million-sample test split.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_repository_root():
    """Support launching Jupyter from the repository root or plotting_code/."""
    current = Path.cwd().resolve()
    if (current / "ml_model.py").is_file() and (current / "plotting_code").is_dir():
        return current
    if current.name == "plotting_code" and (current.parent / "ml_model.py").is_file():
        return current.parent
    raise RuntimeError(
        "Launch this notebook from the PolySurrogate repository root "
        "or from its plotting_code directory."
    )


REPO_ROOT = find_repository_root()
PLOTTING_DIR = REPO_ROOT / "plotting_code"
MAE_DATA_DIR = PLOTTING_DIR / "Data" / "testing_data"
OUTPUT_DIR = PLOTTING_DIR / "Results" / "testing_mae_histograms"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAE_FILES = {
    4: MAE_DATA_DIR / "mae_values_n4.npy",
    6: MAE_DATA_DIR / "mae_values_n6.npy",
    8: MAE_DATA_DIR / "mae_values_n8.npy",
}

VERSION = 9
SAVE_FORMATS = ("png", "pdf", "svg")
SAVE_DPI = 500

X_SCALE = 1e6
X_SCALE_LABEL = r"$\times 10^{-6}$"
NUM_BINS = 35

HIST_COLOR = "skyblue"
HIST_EDGE_COLOR = "black"
HIST_ALPHA = 0.85
HIST_LINEWIDTH = 1.8
MEAN_LINE_COLOR = "red"
MEAN_LINEWIDTH = 2.2

LABEL_FONTSIZE = 22
TICK_FONTSIZE = 18
LEGEND_FONTSIZE = 16
X_LABELPAD = 8
Y_LABELPAD = 8
TICK_PAD = 5

plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
    "lines.linewidth": 2.0,
    "axes.linewidth": 0.9,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "figure.dpi": 150,
    "savefig.dpi": SAVE_DPI,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.unicode_minus": False,
})

missing_files = [path for path in MAE_FILES.values() if not path.is_file()]
if missing_files:
    formatted = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(f"Missing MAE data files:\n{formatted}")

print("MAE data     :", MAE_DATA_DIR)
print("Figure output:", OUTPUT_DIR)

## Load and validate the MAE data

In [ ]:
def load_mae_values(path):
    """Load a one-dimensional MAE array and discard non-finite values."""
    values = np.asarray(np.load(path), dtype=float).ravel()
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError(f"No finite MAE values found in {path}")
    return values


mae_by_node = {
    node_count: load_mae_values(path)
    for node_count, path in MAE_FILES.items()
}

for node_count, values in mae_by_node.items():
    print(
        f"n={node_count}: {values.size:,} samples, "
        f"mean={values.mean():.6e}"
    )

## Generate the histograms

In [ ]:
def style_axis(ax):
    ax.grid(False)
    ax.tick_params(
        axis="both", labelsize=TICK_FONTSIZE, width=0.8, length=3.5,
        direction="out", pad=TICK_PAD,
    )
    for spine in ax.spines.values():
        spine.set_linewidth(2.2)


def save_figure(fig, filename):
    for extension in SAVE_FORMATS:
        path = OUTPUT_DIR / f"{filename}.{extension}"
        dpi = SAVE_DPI if extension == "png" else None
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        print(f"Saved: {path}")


def plot_histogram(node_count, mae_values):
    mae_scaled = mae_values * X_SCALE
    mean_raw = mae_values.mean()

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.hist(
        mae_scaled, bins=NUM_BINS, alpha=HIST_ALPHA, color=HIST_COLOR,
        edgecolor=HIST_EDGE_COLOR, linewidth=HIST_LINEWIDTH,
        label="MAE distribution",
    )
    ax.axvline(
        mean_raw * X_SCALE, color=MEAN_LINE_COLOR, linewidth=MEAN_LINEWIDTH,
        label=f"Mean = {mean_raw:.2e}",
    )
    ax.legend(
        loc="upper right",
        prop={"weight": "bold", "size": LEGEND_FONTSIZE},
        frameon=True, framealpha=0.92, edgecolor="0.75",
        handlelength=1.8, borderpad=0.3, labelspacing=0.2,
    )
    ax.set_xlabel(
        f"Sample wise MAE ({X_SCALE_LABEL})",
        fontsize=LABEL_FONTSIZE, fontweight="bold", labelpad=X_LABELPAD,
    )
    ax.set_ylabel(
        "Sample Count",
        fontsize=LABEL_FONTSIZE, fontweight="bold", labelpad=Y_LABELPAD,
    )
    style_axis(ax)
    fig.tight_layout()

    filename = f"testing_mae_histogram_n{node_count}_v{VERSION}"
    save_figure(fig, filename)
    plt.show()
    plt.close(fig)


for node_count, mae_values in mae_by_node.items():
    print(f"\nPlotting histogram for n={node_count}")
    plot_histogram(node_count, mae_values)

print("\nGenerated all three test-set MAE histograms.")